In [ ]:
"""
Copyright (c) 2021-2024 D-Robotics Corporation

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

     http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
"""

In [ ]:
!cat /proc/meminfo | grep Mem

In [ ]:
# 导入所需要的包
import numpy as np
import cv2
import time
import ctypes
import json
import os

# Hobot DNN library
from hobot_dnn import pyeasy_dnn as dnn

# For displaying images in Jupyter
import matplotlib.pyplot as plt

#### 检查模型，图片和C后处理库路径

In [ ]:
# --- Configuration ---
NOTEBOOK_DIR = '.' 

MODEL_NAME = 'efficientnet_lite0_224x224_nv12.hbm'
MODEL_PATH = os.path.join(NOTEBOOK_DIR, './model', MODEL_NAME)

IMAGE_NAME = 'Scottish_deerhound.JPEG'
IMAGE_PATH = os.path.join(NOTEBOOK_DIR, './data', IMAGE_NAME)

LIB_POSTPROCESS_PATH = '/usr/lib/libpostprocess.so'
# --- End Configuration ---

print(f"Notebook directory: {os.path.abspath(NOTEBOOK_DIR)}")
print(f"Model path: {os.path.abspath(MODEL_PATH)}")
print(f"Image path: {os.path.abspath(IMAGE_PATH)}")
print(f"Postprocess library path: {LIB_POSTPROCESS_PATH}")

# Check if files exist (optional but good for early feedback)
if not os.path.exists(MODEL_PATH):
    print(f"WARNING: Model file not found at {MODEL_PATH}")
if not os.path.exists(IMAGE_PATH):
    print(f"WARNING: Image file not found at {IMAGE_PATH}")
if not os.path.exists(LIB_POSTPROCESS_PATH):
    print(f"WARNING: Shared library not found at {LIB_POSTPROCESS_PATH}")

#### 导入C后处理库以相关工具

In [ ]:
from python.hobot_structures import (
    hbDNNTensor_t,
    ClassificationPostProcessInfo_t
)
try:
    libpostprocess = ctypes.CDLL(LIB_POSTPROCESS_PATH)
    print(f"Successfully loaded {LIB_POSTPROCESS_PATH}")
except OSError as e:
    print(f"Error loading {LIB_POSTPROCESS_PATH}: {e}")
    print("Please ensure the library path is correct, it exists, and it's compatible with your system.")

get_Postprocess_result = libpostprocess.ClassificationPostProcess
get_Postprocess_result.argtypes = [
    ctypes.POINTER(ClassificationPostProcessInfo_t)]
get_Postprocess_result.restype = ctypes.c_char_p

from python.utils import bgr2nv12_opencv, print_properties, get_hw

def display_image(image_cv, title="Image", size=(5, 5)):
    """Displays an OpenCV image (BGR) using Matplotlib (RGB)."""
    # Convert BGR to RGB for Matplotlib
    image_rgb = cv2.cvtColor(image_cv, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=size)
    plt.imshow(image_rgb)
    plt.title(title)
    plt.axis('off') # Hide axes
    plt.show()

#### 导入HBM模型并检查

In [ ]:
print(f"Loading DNN model from: {MODEL_PATH}")
# This step requires the Hobot DNN environment and compatible hardware/emulator
try:
    # dnn.load can return a list of models if the .hbm file contains multiple
    loaded_models = dnn.load(MODEL_PATH)
    if not loaded_models:
        raise ValueError("dnn.load returned an empty list. Model loading failed.")
    hobot_model = loaded_models[0] # Assuming we are interested in the first model
    print("DNN Model loaded successfully.")

    print("\n" + "=" * 10, "Model Input[0] Properties", "=" * 10)
    print_properties(hobot_model.inputs[0].properties)
    print(f"Input[0] name: {hobot_model.inputs[0].name}")

    print("\n" + "=" * 10, "Model Output[0] Properties", "=" * 10)
    if hobot_model.outputs: # Check if there are any outputs
        print_properties(hobot_model.outputs[0].properties)
        print(f"Output[0] name: {hobot_model.outputs[0].name}")
    else:
        print("Model has no defined outputs in the loaded structure.")

except Exception as e:
    print(f"An error occurred during model loading or property inspection: {e}")
    print("Ensure the Hobot DNN environment is active and the model path is correct.")

#### 导入图片并检查

In [ ]:
print(f"Loading image from: {IMAGE_PATH}")
original_bgr_image = cv2.imread(IMAGE_PATH)

if original_bgr_image is None:
    print(f"Failed to load image at {IMAGE_PATH}. Please check the path.")
    # raise FileNotFoundError(f"Image not found at {IMAGE_PATH}") # Stop execution
else:
    print(f"Original image loaded. Shape: {original_bgr_image.shape}")
    display_image(original_bgr_image, title="Original Image")

    # Get target H, W from model input properties
    model_input_height, model_input_width = get_hw(hobot_model.inputs[0].properties)
    target_dim = (model_input_width, model_input_height) # OpenCV uses (width, height)

    print(f"Resizing image to model input dimensions: {target_dim}")
    resized_bgr_image = cv2.resize(original_bgr_image, target_dim, interpolation=cv2.INTER_AREA)
    display_image(resized_bgr_image, title=f"Resized Image ({model_input_width}x{model_input_height})")

    print("Converting resized BGR image to NV12 format...")
    nv12_image_data = bgr2nv12_opencv(resized_bgr_image)
    print(f"NV12 data ready. Shape: {nv12_image_data.shape}") # Should be 1D array

#### 进行模型推理

In [ ]:
if 'nv12_image_data' in locals() and nv12_image_data is not None:
    print("Running model inference...")
    start_time_inference = time.time()
    try:
        outputs = hobot_model.forward(nv12_image_data)
        inference_time = time.time() - start_time_inference
        print(f"Inference complete in {inference_time:.4f} seconds.")
        print(f"Received {len(outputs)} output tensor(s) from the model.")
    except Exception as e:
        print(f"An error occurred during model inference: {e}")
        outputs = [] # Ensure dnn_outputs exists but is empty on error
else:
    print("Skipping inference as preprocessed image data is not available.")
    outputs = []

#### 对推理结果进行处理

In [ ]:
if outputs: # Proceed only if inference was successful
    print("\nStarting post-processing using the C library...")
    t0 = time.time()
    # 获取结构体信息
    classification_postprocess_info = ClassificationPostProcessInfo_t()
    classification_postprocess_info.height = model_input_height
    classification_postprocess_info.width = model_input_width
    org_height, org_width = original_bgr_image.shape[0:2]
    classification_postprocess_info.ori_height = org_height
    classification_postprocess_info.ori_width = org_width
    classification_postprocess_info.score_threshold = 0.3
    classification_postprocess_info.nms_threshold = 0
    classification_postprocess_info.nms_top_k = 5
    classification_postprocess_info.is_pad_resize = 0
    classification_postprocess_info.use_softmax = True

    output_tensors = (hbDNNTensor_t * len(hobot_model.outputs))()
    
    for i in range(len(hobot_model.outputs)):
        if (len(outputs[i].properties.scale_data) == 0):
            output_tensors[i].properties.quantiType = 0
            output_tensors[i].sysMem.virAddr = ctypes.cast(
                outputs[i].buffer.ctypes.data_as(ctypes.POINTER(ctypes.c_float)), ctypes.c_void_p)
        else:
            output_tensors[i].properties.quantiType = 1
            output_tensors[i].properties.scale.scaleData = outputs[i].properties.scale_data.ctypes.data_as(
                ctypes.POINTER(ctypes.c_float))
            output_tensors[i].sysMem.virAddr = ctypes.cast(
                outputs[i].buffer.ctypes.data_as(ctypes.POINTER(ctypes.c_int32)), ctypes.c_void_p)

        for j in range(len(outputs[i].properties.shape)):
            output_tensors[i].properties.validShape.numDimensions = len(
                outputs[i].properties.shape)
            output_tensors[i].properties.validShape.dimensionSize[j] = outputs[i].properties.shape[j]

        libpostprocess.ClassificationDoProcess(
            output_tensors[i], ctypes.pointer(classification_postprocess_info), i)

    result_str = get_Postprocess_result(
        ctypes.pointer(classification_postprocess_info))
    result_str = result_str.decode('utf-8')
    
    t1 = time.time()
    print("postprocess time is :", (t1 - t0))



#### 显示推理结果

In [ ]:
if result_str:
    # This is specific to the output format of your libpostprocess.so
    json_offset = 25
    if len(result_str) > json_offset:
        json_payload_string = result_str[json_offset:]
    else:
        print(f"Warning: Result string is shorter than offset {json_offset}. Using full string.")
        json_payload_string = result_str

    print(f"\nAttempting to parse JSON from: \"{json_payload_string[:100]}...\"") # Print a snippet for preview

    try:
        parsed_results = json.loads(json_payload_string)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON: {e}")
        print("Failed JSON payload:", json_payload_string)
        parsed_results = [] # Set to empty list to avoid error in loop below

    print("\n" + "=" * 10, "Classification Results", "=" * 10)
    if parsed_results and isinstance(parsed_results, list):
        for item_index, result_item in enumerate(parsed_results):
            prob = result_item.get('prob', 'N/A')
            label_id = result_item.get('label', 'N/A')
            class_name = result_item.get('class_name', 'N/A')

            print(f"Result {item_index + 1}: Class ID: {label_id}, Confidence: {prob:.4f}, Name: {class_name}")

        # Optionally, annotate the image with the top result
        if parsed_results and original_bgr_image is not None:
            top_result = parsed_results[0]
            top_name = top_result.get('class_name', 'Unknown')
            top_prob = top_result.get('prob', 0)
            display_text = f"{top_name} ({top_prob:.2f})"

            origin_image = original_bgr_image.copy()
            cv2.putText(origin_image, display_text, (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2, cv2.LINE_AA)
            display_image(origin_image, title="Top Classification Result on Resized Image")

    elif parsed_results: 
        print("Parsed JSON data is not in the expected list format:", parsed_results)
    else:
        print("No valid classification results to display.")
else:
    print("No result string from post-processing to parse.")

print("\n--- Script execution finished ---")

In [ ]:
!hrt_model_exec perf --model_file ./model/efficientnet_lite0_224x224_nv12.hbm \
                    --core_id=0 \
                    --frame_count=200 \
                    --perf_time=0 \
                    --thread_num=3